# Step 1 — Build the clean project-level base table
**CSI Competitive Sweet-Spot Analysis** · read `2026-06-04_00_METHODOLOGY.md` first.

The raw master is **one row per (project × bidder) bid**. Almost everything we
want to reason about lives at the **project** level, so this notebook collapses
it to **one row per project** and does the cleaning every later step depends on:

1. drop 7 rows with no project id, de-duplicate repeated (project, bidder) rows
2. parse the comma-formatted tonnage strings (`"20,626"` → `20626.0`)
3. pull **CSI's own bid row** to get how CSI did (rank, price quality)
4. build the **material-mix proportions** (the "recipe" — see methodology §4)

> ⚠️ **Win-rate correction baked in here:** CSI wins **117 / 603 (19%)** of jobs
> it bids — *not* 70%. The old "421" was a row count (≈104 won projects × ~4
> bidder rows). We derive "CSI won" from CSI's own row ranking 1st.

In [1]:
from pathlib import Path
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)  # spurious BLAS matmul noise on this machine
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 40)

# --- Absolute paths (edit BASE if you move the project) -----------------------
BASE   = Path("/Users/t/Library/Mobile Documents/com~apple~CloudDocs/00 MSBA + Coding/00_summer 2026/00_CSI_ELP/01_Summer_2026")
MASTER = BASE / "2026-06-03_csi_bid_master.csv"          # raw source (read-only)
OUT    = BASE / "06_sweetspot" / "outputs"               # intermediates + figures
OUT.mkdir(parents=True, exist_ok=True)
MAT = ["MIX_TN","GRADE_CY","BASE_TN","SHLDR_TN","MILL_SY","CRUSH_TN","STOCK_TN"]
print("paths OK |", MASTER.name, "exists:", MASTER.exists())

paths OK | 2026-06-03_csi_bid_master.csv exists: True


### Helpers
- `to_num` strips the thousands-commas before converting (this is why the
  earlier inventory wrongly thought tonnage was "capped at 999" — it choked on
  the commas).
- `to_bool` maps the `"TRUE"/"FALSE"` **strings** to real booleans.
- `first_nonblank` picks a project's value from its bidder rows (project-level
  fields are constant across a project's bidders, so this is loss-free).

In [2]:
def to_num(s):   # "20,626" -> 20626.0 ; blank -> NaN
    return pd.to_numeric(s.str.replace(",", "", regex=False).str.strip().replace("", np.nan), errors="coerce")
def to_bool(s):
    return s.str.strip().str.upper().map({"TRUE": True, "FALSE": False})
def first_nonblank(s):
    nb = s[s.astype(str).str.strip() != ""]
    return nb.iloc[0] if len(nb) else ""

### Load + clean rows

In [3]:
df = pd.read_csv(MASTER, dtype=str, keep_default_na=False)
print("loaded:", df.shape)
df = df[df["Contract_Id"].str.strip() != ""].copy()                 # drop null ids
df = df.drop_duplicates(subset=["Contract_Id","Bidder_Clean_Name"], keep="first")
for c in MAT: df[c+"_num"] = to_num(df[c])                          # parse tonnage
print("after clean:", df.shape, "| unique projects:", df["Contract_Id"].nunique())

loaded: (6916, 46)
after clean: (6854, 53) | unique projects: 1591


### CSI's own outcome row
We isolate CSI's bidder row per project to read **how CSI did**: its bid rank,
the price-quality bucket (`Bid_Quality`), and the competitive margin.
`CSI_Won` = CSI's row ranked 1st (the authoritative win flag we use downstream;
the project-level `CSI_Won_Project` flag disagrees on ~13 projects).

In [4]:
df["_is_csi"] = to_bool(df["Is_CSI"])
csi = (df[df["_is_csi"] == True]
       [["Contract_Id","Bid_Quality","Margin_Pct","Bid_Rank","Bid_Amount","Is_Winner"]]
       .rename(columns={"Bid_Quality":"CSI_Bid_Quality","Margin_Pct":"CSI_Margin_Pct",
                        "Bid_Rank":"CSI_Row_Bid_Rank","Bid_Amount":"CSI_Row_Bid_Amount",
                        "Is_Winner":"CSI_Row_Is_Winner"})
       .drop_duplicates("Contract_Id"))
csi["CSI_Won"] = to_bool(csi["CSI_Row_Is_Winner"])
print("CSI bid on", len(csi), "projects | CSI won (row-derived):", int(csi["CSI_Won"].sum()))

CSI bid on 603 projects | CSI won (row-derived): 117


### Collapse to one row per project + derive the material recipe

In [5]:
proj_cols = ["Project_Name","State","County","City","Latitude","Longitude",
    "Standard_Region","CSI_Region","Date","Bid_Year","Bid_Quarter",
    "Project_Size_Dollars","Num_Valid_Bidders","Num_Bidders","Market_Tightness_Category",
    "Winner_Amount","Second_Place_Amount","CSI_Participated","CSI_Won_Project","CSI_Bid_Rank",
    "BOX","PREVAILING_WAGES"] + [c+"_num" for c in MAT]
proj = df.groupby("Contract_Id", sort=False)[proj_cols].agg(first_nonblank).reset_index()
proj = proj.merge(csi, on="Contract_Id", how="left")

# typed numerics
for c in ["Project_Size_Dollars","Num_Valid_Bidders","Num_Bidders","Winner_Amount",
          "Second_Place_Amount","CSI_Bid_Rank","CSI_Margin_Pct"] + [c+"_num" for c in MAT]:
    proj[c] = pd.to_numeric(proj[c].astype(str).str.replace(",","",regex=False).str.strip().replace("", np.nan), errors="coerce")
proj["CSI_Participated"] = to_bool(proj["CSI_Participated"])
proj["CSI_Won_Project"]  = to_bool(proj["CSI_Won_Project"])
proj["BOX"] = (proj["BOX"].str.strip().str.upper() == "X")
proj["PREVAILING_WAGES"] = proj["PREVAILING_WAGES"].str.strip().str.upper().map({"YES":True,"NO":False})

# --- material recipe: proportions that sum to 1 (scale-free shape) ---
num_cols = [c+"_num" for c in MAT]
present = proj[num_cols].notna() & (proj[num_cols] > 0)
proj["n_mat_types"] = present.sum(axis=1)            # how many trades present
proj["has_quantity"] = proj["n_mat_types"] > 0
tot = proj[num_cols].fillna(0).sum(axis=1)
for c in MAT:
    proj["prop_"+c] = np.where(tot > 0, proj[c+"_num"].fillna(0)/tot, np.nan)
proj["total_quantity"] = np.where(tot > 0, tot, np.nan)
proj.head(3)

,Contract_Id,Project_Name,State,County,City,Latitude,Longitude,Standard_Region,CSI_Region,Date,Bid_Year,Bid_Quarter,Project_Size_Dollars,Num_Valid_Bidders,Num_Bidders,Market_Tightness_Category,Winner_Amount,Second_Place_Amount,CSI_Participated,CSI_Won_Project,...,SHLDR_TN_num,MILL_SY_num,CRUSH_TN_num,STOCK_TN_num,CSI_Bid_Quality,CSI_Margin_Pct,CSI_Row_Bid_Rank,CSI_Row_Bid_Amount,CSI_Row_Is_Winner,CSI_Won,n_mat_types,has_quantity,prop_MIX_TN,prop_GRADE_CY,prop_BASE_TN,prop_SHLDR_TN,prop_MILL_SY,prop_CRUSH_TN,prop_STOCK_TN,total_quantity
0,1,PEACE GARDEN INTL AIRPORT PAVEMENT REHAB EC22-...,North Dakota,Rolette County,Bismarck,48.99747682,-100.0442833,Eastern ND,,4/30/24,2024,2,2920773.0,3,3,Loose Market,2920773.0,3216031.0,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,SANITARY SEWER LIFT STATION REHABILITATION,North Dakota,Bottineau County,Antler,48.97299497,-101.2796076,Western ND,Western ND,2/7/24,2024,1,249696.0,4,4,Loose Market,249696.0,342000.0,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,107TH ST. NW BRIDGE REPLACEMENT,North Dakota,Burke County,Columbus,48.96691101,-102.8551016,Western ND,Western ND,3/19/24,2024,1,518262.0,4,3,Moderate Market,518262.0,701902.2,True,False,...,NaN,NaN,NaN,NaN,Unattainable,0.3543,2,701902.2,FALSE,False,0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Verify against known anchors, then save

In [6]:
nbid = int((proj["CSI_Participated"]==True).sum())
print(f"projects            : {len(proj)}  (expect 1591)")
print(f"CSI bid / no-bid    : {nbid} / {int((proj['CSI_Participated']==False).sum())}  (expect 603 / 988)")
print(f"CSI won (row)       : {int((proj['CSI_Won']==True).sum())}  | flag: {int((proj['CSI_Won_Project']==True).sum())}")
print(f"win rate when bidding: {int((proj['CSI_Won']==True).sum())/nbid*100:.1f}%  (NOT 70%)")
print(f"has_quantity        : {proj['has_quantity'].mean()*100:.1f}%  | MIX median: {proj['MIX_TN_num'].median():,.0f}")
proj.to_csv(OUT / "2026-06-04_project_base.csv", index=False)
print("saved ->", "2026-06-04_project_base.csv")

projects            : 1591  (expect 1591)
CSI bid / no-bid    : 603 / 988  (expect 603 / 988)
CSI won (row)       : 117  | flag: 104
win rate when bidding: 19.4%  (NOT 70%)
has_quantity        : 86.5%  | MIX median: 5,872
saved -> 2026-06-04_project_base.csv


**Takeaway:** we now have `project_base.csv` — one tidy row per project with
clean numerics, CSI's outcome, and the material recipe. Every later notebook
starts from this file.